## Install & Import Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, roc_curve
)

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')
print('All libraries imported ✓')

## Load Data



In [ ]:
df_loan = pd.read_csv('Assignment3-Loan-Dataset.csv')
df_unknown = pd.read_csv('Assignment3-Unknown-Dataset.csv')

print(f'Loan:    {df_loan.shape[0]:,} rows × {df_loan.shape[1]} cols')
print(f'Unknown: {df_unknown.shape[0]:,} rows × {df_unknown.shape[1]} cols')

In [ ]:
df_loan.head()

In [ ]:
df_loan.info()

In [ ]:
missing = df_loan.isnull().sum()
missing_pct = (missing / len(df_loan) * 100).round(2)
mi = pd.DataFrame({'Count': missing, '%': missing_pct})
print(mi[mi['Count']>0].sort_values('%', ascending=False))

## Preprocessing

In [ ]:
# Drop date column
df_loan = df_loan.drop(columns=['application_date'], errors='ignore')
df_unknown = df_unknown.drop(columns=['application_date'], errors='ignore')

# Identify types
target = 'loan_default'
num_cols = [c for c in df_loan.select_dtypes(include=['int64','float64']).columns if c != target]
cat_cols = df_loan.select_dtypes(include=['object']).columns.tolist()
print(f'Numerical ({len(num_cols)}): {num_cols}')
print(f'Categorical ({len(cat_cols)}): {cat_cols}')

In [ ]:
# Impute missing values
medians = {}
for col in num_cols:
    medians[col] = df_loan[col].median()
    df_loan[col] = df_loan[col].fillna(medians[col])
    df_unknown[col] = df_unknown[col].fillna(medians[col])

for col in cat_cols:
    df_loan[col] = df_loan[col].fillna('Unknown')
    df_unknown[col] = df_unknown[col].fillna('Unknown')

print(f'NaN remaining — loan: {df_loan.isnull().sum().sum()}, unknown: {df_unknown.isnull().sum().sum()}')

In [ ]:
# One-hot encode (both datasets together)
df_loan['_src'] = 'loan'
df_unknown['_src'] = 'unk'
df_unknown[target] = np.nan

combined = pd.concat([df_loan, df_unknown], ignore_index=True)
combined = pd.get_dummies(combined, columns=cat_cols, drop_first=True, dtype=int)

df_loan_enc = combined[combined['_src']=='loan'].drop(columns=['_src']).reset_index(drop=True)
df_unknown_enc = combined[combined['_src']=='unk'].drop(columns=['_src', target]).reset_index(drop=True)

print(f'Loan encoded:    {df_loan_enc.shape}')
print(f'Unknown encoded: {df_unknown_enc.shape}')
print(f'Columns match: {set(df_loan_enc.columns)-{target} == set(df_unknown_enc.columns)}')

## EDA

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df_loan_enc[target].value_counts()
axes[0].bar(['No Default (0)','Default (1)'], counts.values, color=['steelblue','salmon'], edgecolor='black')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+300, f'{v:,}\n({v/len(df_loan_enc)*100:.1f}%)', ha='center', fontweight='bold')
axes[0].set_title('Target Distribution'); axes[0].set_ylabel('Count')
axes[1].pie(counts.values, labels=['No Default','Default'], autopct='%1.1f%%', colors=['steelblue','salmon'])
axes[1].set_title('Class Proportion')
plt.tight_layout(); plt.show()
print(f'Imbalance ratio: {counts[0]/counts[1]:.2f}:1')
print(f'Majority baseline: {counts.max()/len(df_loan_enc):.1%}')

In [ ]:
# Top correlations with target
corr = df_loan_enc.select_dtypes('number').corr()[target].drop(target).sort_values(key=abs, ascending=False)
top15 = corr.head(15)
plt.figure(figsize=(10,6))
plt.barh(top15.index, top15.values, color=['salmon' if v>0 else 'steelblue' for v in top15.values], edgecolor='black')
plt.xlabel('Correlation with loan_default'); plt.title('Top 15 Features by Correlation')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
# Key feature distributions by class
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, feat in zip(axes.flatten(), ['credit_score','income','debt_to_income_pct','loan_amount']):
    if feat in df_loan_enc.columns:
        for cls, col, lab in [(0,'steelblue','No Default'),(1,'salmon','Default')]:
            ax.hist(df_loan_enc.loc[df_loan_enc[target]==cls, feat], bins=40, alpha=0.5, color=col, label=lab)
        ax.set_title(f'{feat} by Default Status'); ax.legend()
plt.tight_layout(); plt.show()

## Feature Engineering

In [ ]:
# Create ratio features on BOTH datasets
for df in [df_loan_enc, df_unknown_enc]:
    df['loan_to_income'] = df['loan_amount'] / df['income'].replace(0, np.nan)
    df['loan_to_income'] = df['loan_to_income'].fillna(df['loan_to_income'].median())
    df['expense_to_income'] = df['monthly_expenses'] / df['income'].replace(0, np.nan)
    df['expense_to_income'] = df['expense_to_income'].fillna(df['expense_to_income'].median())
    df['balance_to_loan'] = df['account_balance'] / df['loan_amount'].replace(0, np.nan)
    df['balance_to_loan'] = df['balance_to_loan'].fillna(df['balance_to_loan'].median())

print(f'Features after engineering: {df_loan_enc.shape[1]}')

---
# Train/Test Split & Scaling

In [ ]:
# Prepare X, y
feature_cols = [c for c in df_loan_enc.columns if c != target]
X = df_loan_enc[feature_cols]
y = df_loan_enc[target].astype(int)

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale for KNN, SVM, NN
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} | Features: {X_train.shape[1]}')
print(f'Train default rate: {y_train.mean():.3f} | Test: {y_test.mean():.3f}')

##  Evaluation Function

This function is used by all 5 models to compute metrics and show plots.

In [ ]:
# Store all results here
all_results = {}

def evaluate(model, X_tr, y_tr, X_te, y_te, name, use_decision_fn=False):
    """Evaluate a model and store results."""
    yp = model.predict(X_te)
    if use_decision_fn:
        y_scores = model.decision_function(X_te)
    else:
        y_scores = model.predict_proba(X_te)[:,1]
    
    tr_acc = accuracy_score(y_tr, model.predict(X_tr))
    te_acc = accuracy_score(y_te, yp)
    prec = precision_score(y_te, yp)
    rec  = recall_score(y_te, yp)
    f1   = f1_score(y_te, yp)
    f1m  = f1_score(y_te, yp, average='macro')
    auc  = roc_auc_score(y_te, y_scores)
    cm   = confusion_matrix(y_te, yp)
    
    print(f'\n{"="*55}')
    print(f'  {name}')
    print(f'{"="*55}')
    print(f'  Train Acc : {tr_acc:.4f}')
    print(f'  Test Acc  : {te_acc:.4f}')
    print(f'  Gap       : {tr_acc-te_acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1        : {f1:.4f}')
    print(f'  F1-macro  : {f1m:.4f}')
    print(f'  ROC-AUC   : {auc:.4f}')
    print(classification_report(y_te, yp, target_names=['No Default','Default'], digits=4))
    
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay.from_predictions(y_te, yp, display_labels=['No Def','Default'], cmap='Blues', ax=axes[0])
    axes[0].set_title(f'{name} — Confusion Matrix')
    if not use_decision_fn:
        RocCurveDisplay.from_predictions(y_te, y_scores, ax=axes[1], name=name)
    else:
        fpr, tpr, _ = roc_curve(y_te, y_scores)
        axes[1].plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})')
        axes[1].legend()
    axes[1].plot([0,1],[0,1],'k--'); axes[1].set_title(f'{name} — ROC Curve')
    plt.tight_layout(); plt.show()
    
    all_results[name] = {
        'accuracy': te_acc, 'precision': prec, 'recall': rec,
        'f1': f1, 'f1_macro': f1m, 'auc': auc, 'cm': cm,
        'y_scores': y_scores, 'y_pred': yp
    }
    return all_results[name]

print('Evaluation function ready ✓')

---
# Decision Tree



In [ ]:
# Baseline (unrestricted)
dt_base = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt_base.fit(X_train, y_train)
print(f'Baseline: depth={dt_base.get_depth()}, leaves={dt_base.get_n_leaves()}')
print(f'Train acc: {accuracy_score(y_train, dt_base.predict(X_train)):.4f}')
print(f'Test acc:  {accuracy_score(y_test, dt_base.predict(X_test)):.4f}')


In [ ]:
# Hyperparameter tuning
dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    {'criterion':['gini','entropy'], 'max_depth':[5,7,9,11,13],
     'min_samples_leaf':[1,5,10,15,25], 'min_samples_split':[2,10,20]},
    cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='f1_macro', n_jobs=-1, verbose=1
)
dt_grid.fit(X_train, y_train)
print(f'\nBest params: {dt_grid.best_params_}')
print(f'Best CV F1-macro: {dt_grid.best_score_:.4f}')

In [ ]:
# Evaluate tuned DT
dt_tuned = dt_grid.best_estimator_
evaluate(dt_tuned, X_train, y_train, X_test, y_test, 'Decision Tree')

In [ ]:
# Tree visualisation
plt.figure(figsize=(24, 10))
plot_tree(dt_tuned, max_depth=3, feature_names=feature_cols,
          class_names=['No Default','Default'], filled=True, rounded=True, fontsize=8, proportion=True)
plt.title(f'Decision Tree (top 3 of {dt_tuned.get_depth()} levels)'); plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
imp = pd.Series(dt_tuned.feature_importances_, index=feature_cols).sort_values(ascending=True).tail(15)
plt.figure(figsize=(10,6))
plt.barh(imp.index, imp.values, color=['salmon' if v>0.05 else 'steelblue' for v in imp.values], edgecolor='black')
plt.xlabel('Importance'); plt.title('DT — Top 15 Features'); plt.tight_layout(); plt.show()

---
# K-Nearest Neighbours (KNN)



In [ ]:
# Baseline
knn_base = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn_base.fit(X_train_s, y_train)
print(f'Baseline KNN (k=5): test acc = {accuracy_score(y_test, knn_base.predict(X_test_s)):.4f}')

In [ ]:
# Tuning
knn_grid = GridSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    {'n_neighbors':[5,11,21,31,51], 'metric':['euclidean','manhattan'], 'weights':['uniform','distance']},
    cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='f1_macro', n_jobs=-1, verbose=1
)
knn_grid.fit(X_train_s, y_train)
print(f'\nBest params: {knn_grid.best_params_}')
print(f'Best CV F1-macro: {knn_grid.best_score_:.4f}')

In [ ]:
knn_tuned = knn_grid.best_estimator_
evaluate(knn_tuned, X_train_s, y_train, X_test_s, y_test, 'KNN')

In [ ]:
# K-value analysis
ks = [3,5,7,11,15,21,31,51]
k_scores = []
for k in ks:
    knn_k = KNeighborsClassifier(n_neighbors=k, metric='manhattan', weights='distance', n_jobs=-1)
    knn_k.fit(X_train_s, y_train)
    k_scores.append(f1_score(y_test, knn_k.predict(X_test_s), average='macro'))
plt.figure(figsize=(8,4))
plt.plot(ks, k_scores, 'o-', color='steelblue', lw=2, ms=7)
plt.xlabel('k'); plt.ylabel('F1-macro'); plt.title('KNN: F1 vs k'); plt.grid(True); plt.tight_layout(); plt.show()

---
# Random Forest



In [ ]:
# Baseline
rf_base = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_base.fit(X_train, y_train)
print(f'Baseline RF: train={accuracy_score(y_train, rf_base.predict(X_train)):.4f}, test={accuracy_score(y_test, rf_base.predict(X_test)):.4f}')

In [ ]:
# Tuning
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    {'n_estimators':[100,200], 'max_depth':[9,12,15,None], 'max_features':['sqrt',0.5], 'min_samples_leaf':[1,5,10]},
    cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='f1_macro', n_jobs=-1, verbose=1
)
rf_grid.fit(X_train, y_train)
print(f'\nBest params: {rf_grid.best_params_}')
print(f'Best CV F1-macro: {rf_grid.best_score_:.4f}')

In [ ]:
rf_tuned = rf_grid.best_estimator_
evaluate(rf_tuned, X_train, y_train, X_test, y_test, 'Random Forest')

In [ ]:
# Feature importance
rf_imp = pd.Series(rf_tuned.feature_importances_, index=feature_cols).sort_values(ascending=True).tail(15)
plt.figure(figsize=(10,6))
plt.barh(rf_imp.index, rf_imp.values, color=['salmon' if v>0.05 else 'steelblue' for v in rf_imp.values], edgecolor='black')
plt.xlabel('Importance'); plt.title('RF — Top 15 Features'); plt.tight_layout(); plt.show()

---
# Support Vector Machine (SVM)



In [ ]:
# Baseline
svm_base = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42)
svm_base.fit(X_train_s, y_train)
print(f'Baseline SVM: test acc = {accuracy_score(y_test, svm_base.predict(X_test_s)):.4f}')

In [ ]:
# Tuning on 8k subsample (SVM is O(n²))
X_sub, y_sub = resample(X_train_s, y_train, n_samples=8000, stratify=y_train, random_state=42)

svm_grid = GridSearchCV(
    SVC(class_weight='balanced', random_state=42),
    {'C':[0.1,1.0,2.0,5.0], 'kernel':['rbf','linear'], 'gamma':['scale']},
    cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='f1_macro', n_jobs=-1, verbose=1
)
svm_grid.fit(X_sub, y_sub)
print(f'\nBest params: {svm_grid.best_params_}')
print(f'Best CV F1-macro: {svm_grid.best_score_:.4f}')

In [ ]:
# Train final SVM on FULL data with best params
svm_tuned = SVC(**svm_grid.best_params_, class_weight='balanced', random_state=42)
svm_tuned.fit(X_train_s, y_train)
evaluate(svm_tuned, X_train_s, y_train, X_test_s, y_test, 'SVM', use_decision_fn=True)

---
# Neural Network 



In [ ]:
# Baseline
nn_base = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=200, random_state=42)
nn_base.fit(X_train_s, y_train)
print(f'Baseline NN: train={accuracy_score(y_train, nn_base.predict(X_train_s)):.4f}, test={accuracy_score(y_test, nn_base.predict(X_test_s)):.4f}, iters={nn_base.n_iter_}')

In [ ]:
# Tuning
nn_grid = GridSearchCV(
    MLPClassifier(solver='adam', max_iter=300, random_state=42, early_stopping=True, validation_fraction=0.1),
    {'hidden_layer_sizes':[(64,32),(128,64),(64,)], 'activation':['relu','tanh'], 'alpha':[0.0001,0.001,0.01]},
    cv=StratifiedKFold(3, shuffle=True, random_state=42),
    scoring='f1_macro', n_jobs=-1, verbose=1
)
nn_grid.fit(X_train_s, y_train)
print(f'\nBest params: {nn_grid.best_params_}')
print(f'Best CV F1-macro: {nn_grid.best_score_:.4f}')

In [ ]:
nn_tuned = nn_grid.best_estimator_
evaluate(nn_tuned, X_train_s, y_train, X_test_s, y_test, 'Neural Network')

In [ ]:
# Loss curve
plt.figure(figsize=(8,4))
plt.plot(nn_tuned.loss_curve_, color='steelblue', lw=2)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'NN Loss Curve (stopped at epoch {nn_tuned.n_iter_})')
plt.grid(True); plt.tight_layout(); plt.show()

---
# Model Comparison 

In [ ]:
# Build comparison table
comp = pd.DataFrame(all_results).T
comp_display = comp[['accuracy','precision','recall','f1','f1_macro','auc']].round(4)
comp_display.columns = ['Accuracy','Precision','Recall','F1','F1-macro','ROC-AUC']
print('=== ALL MODELS — TEST SET PERFORMANCE ===')
print(comp_display.to_string())
print(f'\nBest by AUC:     {comp_display["ROC-AUC"].idxmax()} ({comp_display["ROC-AUC"].max():.4f})')
print(f'Best by F1-macro: {comp_display["F1-macro"].idxmax()} ({comp_display["F1-macro"].max():.4f})')
print(f'Best by Accuracy: {comp_display["Accuracy"].idxmax()} ({comp_display["Accuracy"].max():.4f})')

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grouped bar chart
metrics = ['Accuracy','Precision','Recall','F1','F1-macro','ROC-AUC']
x = np.arange(len(metrics))
w = 0.15
colors = ['#f78166','#7ee787','#d2a8ff','#ff7b72','#ffa657']
for i, (name, row) in enumerate(comp_display.iterrows()):
    axes[0].bar(x + i*w - 2*w, row.values, w, label=name, color=colors[i], edgecolor='black', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics, fontsize=9)
axes[0].set_ylim(0.6, 0.9); axes[0].legend(fontsize=8); axes[0].set_title('All Metrics Comparison')
axes[0].grid(axis='y', alpha=0.3)

# ROC curves
for name, res in all_results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_scores'])
    axes[1].plot(fpr, tpr, lw=2, label=f'{name} (AUC={res["auc"]:.4f})')
axes[1].plot([0,1],[0,1],'k--',alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curves — All Models'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Error analysis
print('=== ERROR BREAKDOWN ===')
print(f'{"Model":<16s} {"TN":>6s} {"FP":>6s} {"FN":>6s} {"TP":>6s} {"Total Err":>10s}')
for name, res in all_results.items():
    cm = res['cm']
    total_err = cm[0,1] + cm[1,0]
    print(f'{name:<16s} {cm[0,0]:6d} {cm[0,1]:6d} {cm[1,0]:6d} {cm[1,1]:6d} {total_err:10d}')

In [ ]:
# Best model selection
best_name = comp_display['ROC-AUC'].idxmax()
best_auc = comp_display.loc[best_name, 'ROC-AUC']
best_f1m = comp_display.loc[best_name, 'F1-macro']
best_acc = comp_display.loc[best_name, 'Accuracy']

print(f'\n{"="*55}')
print(f'  BEST MODEL: {best_name}')
print(f'{"="*55}')
print(f'  ROC-AUC   : {best_auc}')
print(f'  F1-macro  : {best_f1m}')
print(f'  Accuracy  : {best_acc}')